# 문제 4 — 연립방정식과 해의 판정 (가우스 소거·rank·행렬식·역행렬)

풀어야 하는 연립방정식은 다음과 같습니다. 해는 정수로 떨어집니다.

$$\begin{aligned}
2x_1 + 8x_2 + 2x_3 &= 14\\
x_1 + 6x_2 + 4x_3 &= 15\\
2x_1 + 2x_2 + 3x_3 &= 10
\end{aligned}$$

## 이 노트북에서 해야 할 일

| # | 할 일 | 구현할 함수 |
|---|---|---|
| 4-1 | **가우스 소거법을 직접 구현**해 풀고 **각 소거 단계의 첨가행렬을 모두 출력** | `gauss_eliminate` |
| 4-2 | 계수행렬과 첨가행렬의 rank 로 **해의 존재·유일성 판정**, 같을 때와 다를 때의 의미 설명 | `rank` |
| 4-3 | 행렬식과 역행렬을 구하고, **det 이 0 에 가까울 때 왜 위험한지** 설명 | `det`, `inverse_gauss_jordan` |
| 4-4 | **피벗팅 없는** 소거로 첫 피벗이 아주 작을 때의 오차를 재현하고 부분 피벗팅과 비교 | `gauss_eliminate(pivoting=False)` |
| 4-5 | 500x500 에서 `np.linalg.solve` vs `역행렬을 구해서 곱하기` 의 **시간·잔차** 비교 | — |

In [ ]:
import os

# 벤치마크 공정성을 위해 numpy import 전에 BLAS 스레드 수를 1 로 고정한다.
# 500x500 정도 크기에서는 실제 연산보다 스레드 풀 생성/동기화 오버헤드가 더 커서
# 알고리즘 차이가 통째로 가려질 수 있다.
for _var in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "OMP_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_var] = "1"

import inspect
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.vectors import det, gauss_eliminate, inverse_gauss_jordan, rank, row_echelon

rng = np.random.default_rng(42)
np.set_printoptions(precision=6, suppress=True)

for _f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]:
    if _f in {f.name for f in __import__("matplotlib").font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _f
        break
plt.rcParams["axes.unicode_minus"] = False


def check(label, condition):
    tag = "PASS" if condition else "FAIL"
    print("[" + tag + "] " + label)
    return bool(condition)


# 풀어야 하는 연립방정식
A = np.array([[2.0, 8.0, 2.0],
              [1.0, 6.0, 4.0],
              [2.0, 2.0, 3.0]])
b = np.array([14.0, 15.0, 10.0])
print("A =\n", A, "\n\nb =", b)

## 4-1. 가우스 소거법 — 단계별 첨가행렬

가우스 소거는 **첨가행렬 $[A|b]$ 에 행 연산만 적용**해 상삼각으로 만든 뒤
뒤에서부터 대입해 푸는 방법입니다. 행 연산은 해집합을 바꾸지 않습니다.

부분 피벗팅(partial pivoting)은 각 열에서 **절댓값이 가장 큰 성분을 피벗으로**
올리는 행 교환입니다. 왜 그렇게 하는지는 4-4 에서 직접 확인합니다.

**할 일**

- `src/vectors.py` 의 `gauss_eliminate` 를 구현하세요.
  `verbose=True` 면 **초기 첨가행렬과 각 단계(행 교환 / 소거)의 첨가행렬을 모두 출력**해야 합니다.
- 구현한 소스를 `inspect.getsource` 로 노트북에 출력해 두세요.
- 해를 구하고 `A @ x` 가 `b` 와 같은지, 정수로 떨어지는지 확인하세요.

In [ ]:
print(inspect.getsource(gauss_eliminate))

In [ ]:
# TODO: x, steps = gauss_eliminate(A, b, pivoting=True, verbose=True)

In [ ]:
# TODO: 최종 해, 정수해 여부, A @ x, 잔차 |A x - b| 를 출력하세요.
# TODO: np.linalg.solve(A, b) (# 비교 대상) 와도 비교하세요.

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 해가 정수로 떨어지는가
#   - A @ x == b 인가
#   - np.linalg.solve 와 일치하는가
#   - 소거 단계가 모두 기록되었는가 (len(steps))
#   - 마지막 첨가행렬이 상삼각인가

## 4-2. rank 로 해의 존재·유일성 판정

미지수가 $n$ 개일 때, **루셰-카펠리 정리**로 다음처럼 판정합니다.

| 조건 | 판정 |
|---|---|
| $\mathrm{rank}(A) = \mathrm{rank}([A\,|\,b]) = n$ | 유일해 |
| $\mathrm{rank}(A) = \mathrm{rank}([A\,|\,b]) < n$ | 해가 무한히 많음 (자유변수 $n - \mathrm{rank}$ 개) |
| $\mathrm{rank}(A) < \mathrm{rank}([A\,|\,b])$ | 해 없음 |

**할 일**

- 이 문제의 두 rank 를 구해 판정하세요.
- 두 rank 가 같다는 것이 $b$ 와 $A$ 의 열공간 사이에서 무엇을 뜻하는지,
  다르면 소거 결과에 어떤 행이 나타나는지 **직접 예를 만들어** 확인하세요.
  (예: 1행의 배수인 2행을 가진 행렬에 모순되는 b / 모순 없는 b 를 각각 넣어 보기)

### 두 rank 가 같을 때와 다를 때의 의미

- 같을 때: `___`
- 다를 때: `___`
- 같지만 n 보다 작을 때: `___`

In [ ]:
Ab = np.hstack([A, b.reshape(-1, 1)])

# TODO: rank(A), rank(Ab), 미지수 개수를 출력하고 판정 결과를 출력하세요.
#       np.linalg.matrix_rank (# 검산용) 와도 비교하세요.

In [ ]:
# TODO: 두 rank 가 다른 경우(해 없음)와 같지만 n 보다 작은 경우(해 무한)를
#       직접 만들어 각각 판정과 소거 결과의 마지막 행을 출력하세요.

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - rank(A) 와 rank([A|b]) 가 각각 얼마이고 n 과 같은가 -> 유일해 판정
#   - 직접 구현 rank == np.linalg.matrix_rank
#   - 모순 케이스에서 rank(A) < rank([A|b]) 인가
#   - 무모순 종속 케이스에서 두 rank 가 같고 n 보다 작은가

## 4-3. 행렬식과 역행렬 — 그리고 $\det \approx 0$ 의 위험

행렬식은 행 사다리꼴의 **대각성분 곱 x (-1)^(행 교환 횟수)** 로 구합니다.
역행렬은 가우스-조던 소거 $[A|I] \rightarrow [I|A^{-1}]$ 로 구합니다.

**왜 $\det \approx 0$ 일 때 역행렬이 위험한지** 두 관점에서 설명해 보세요.

- 여인수 공식 $A^{-1} = \frac{1}{\det A}\mathrm{adj}(A)$ 에서 무슨 일이 생기는가
- 소거 관점에서 작은 피벗으로 나눈다는 것은 무엇을 뜻하는가

그리고 한 가지 더 확인해 보세요. $\det(cA)=c^n\det(A)$ 이므로
행렬 전체에 0.01 을 곱하기만 해도 $\det$ 는 크게 작아집니다.
그때 **해의 정확도도 같이 나빠지는지** 실험으로 확인하고,
그렇지 않다면 실무에서는 무엇을 기준으로 삼아야 하는지 조사해 적으세요.
(힌트: `np.linalg.cond`)

### det ≈ 0 이 위험한 이유와 진짜 판정 기준

- 여인수 공식 관점: `___`
- 소거 관점: `___`
- 스케일 실험에서 관찰한 것: `___`
- 실무 판정 기준과 그 의미: `___`

In [ ]:
# TODO: det(A) 와 inverse_gauss_jordan(A) 를 구해 출력하고,
#       np.linalg.det / np.linalg.inv (# 검산용) 와 비교하세요.
# TODO: A @ A^-1 == I 인지, A^-1 @ b 가 4-1 의 해와 같은지 확인하세요.

In [ ]:
# TODO: (1) A 에 스케일 s = 1, 1e-1, 1e-2, 1e-3 을 곱해 가며
#           det / 조건수 / 해의 상대오차를 표로 출력하세요.
# TODO: (2) 진짜 특이행렬에 가까워지는 예 (예: [[1,2],[2,4+eps]]) 로
#           eps 를 줄여 가며 같은 표를 출력하고, (1) 과 무엇이 다른지 보세요.

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 직접 구현 det == np.linalg.det
#   - A @ A^-1 == I, A^-1 @ A == I
#   - 직접 구현 역행렬 == np.linalg.inv
#   - A^-1 @ b == 가우스 소거 해
#   - det(A^-1) == 1/det(A)

## 4-4. 피벗팅 없는 소거의 오차

첫 피벗이 아주 작은 고전적인 예제입니다.

$$\begin{bmatrix}\varepsilon & 1\\ 1 & 1\end{bmatrix}
\begin{bmatrix}x_1\\x_2\end{bmatrix}=\begin{bmatrix}1\\2\end{bmatrix},
\qquad \text{참해}\;\; x_1=\frac{1}{1-\varepsilon},\; x_2=\frac{1-2\varepsilon}{1-\varepsilon}$$

**할 일**

- `pivoting=False` 와 `True` 로 각각 풀어 참해와의 상대오차를 비교하세요 ($\varepsilon=10^{-4}$).
- $\varepsilon$ 을 $10^{-2}$ 에서 $10^{-16}$ 까지 줄여 가며 두 오차를 표로 출력하고,
  로그-로그 그래프로 그리세요.
- 피벗팅이 없을 때 **정확히 어느 연산에서 유효숫자가 날아가는지** 설명하세요.

### 피벗팅 없이 소거하면 무슨 일이 일어나는가

- `___`

In [ ]:
def solve_eps(eps):
    """eps 하나에 대해 (참해, 피벗팅 없음 해, 부분 피벗팅 해, 각각의 상대오차) 를 돌려준다."""
    # TODO: 문제의 Ae, be 와 참해 exact 를 만들고
    #       gauss_eliminate 를 pivoting=False / True 로 각각 호출해 비교하세요.
    raise NotImplementedError("solve_eps 를 구현하세요")


EPS = 1e-4
# TODO: EPS 에서의 결과를 출력하고, eps 를 줄여 가며 표를 출력하세요.

In [ ]:
# TODO: loglog 그래프로 두 방식의 오차를 비교해 그리세요.
#       (x축: eps, y축: 상대오차, ax.invert_xaxis() 를 쓰면 읽기 편합니다)

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - eps=1e-4 에서 피벗팅 없는 쪽의 오차가 더 큰가
#   - 부분 피벗팅 오차는 기계정밀도 수준인가
#   - eps 가 작아질수록 피벗팅 없는 오차가 커지는가
#   - 부분 피벗팅은 eps 와 무관하게 안정적인가
#   - 피벗팅 결과가 np.linalg.solve 와 일치하는가

## 4-5. 500x500 — `solve` vs `역행렬을 구해서 곱하기`

**할 일**

- 500x500 무작위 행렬에서 `np.linalg.solve(A, b)` 와 `np.linalg.inv(A) @ b` 의
  **실행 시간과 잔차**를 각각 측정해 표와 막대그래프로 비교하세요.
- 벤치마크는 **워밍업 1회 후 여러 번 재서 최솟값**을 씁니다.
  (평균 대신 최솟값을 쓰는 이유도 생각해 보세요)
- 조건수가 좋은 행렬에서는 잔차 차이가 잘 안 보입니다.
  조건수를 지정한 행렬을 만들어(예: SVD 형태로 조립) 조건수별 오차를 비교해 보세요.
- 두 방법의 **연산량**이 각각 얼마인지 조사해 표로 정리하고,
  실무에서 어느 쪽을 써야 하는지 근거와 함께 결론을 쓰세요.

### 결론 — 실무에서는 무엇을 쓰는가

- 연산량: `___`
- 정확도: `___`
- 메모리: `___`
- 우변이 여러 개일 때는: `___`
- 예외적으로 역행렬을 명시적으로 구해야 하는 경우: `___`

In [ ]:
N = 500
A_big = rng.standard_normal((N, N))
b_big = rng.standard_normal(N)
REPEAT = 10


def bench(fn, repeat=REPEAT):
    """워밍업 1회 후 repeat 번 재서 최솟값을 쓴다. (그대로 쓰면 됩니다)"""
    fn()                                     # 워밍업 (LAPACK 초기화, 캐시 적재)
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        out = fn()
        best = min(best, time.perf_counter() - t0)
    return best, out


# TODO: 두 방법의 시간과 잔차를 측정해 표로 출력하세요.
#       메모리 사용량 차이도 계산해 함께 적으면 좋습니다.

In [ ]:
# TODO: 조건수를 지정한 행렬을 만들어 조건수별 해의 상대오차를 비교 출력하세요.
#       (힌트: np.linalg.qr 로 만든 직교행렬 두 개와 np.logspace 로 만든 특이값으로 조립)

In [ ]:
# TODO: 시간 / 잔차 막대그래프 2개를 그리세요.

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 두 방법의 해가 사실상 같은가
#   - 어느 쪽이 빠른가
#   - 잔차 비교 결과가 예상과 맞는가
#   - 두 해 모두 유효한가 (잔차가 충분히 작은가)

## 답안 템플릿 정리

In [ ]:
summary = """
1. 가우스 소거 단계별 첨가행렬: 4-1 셀 출력 참조
   최종 해: x = ___

2. rank A: ___ / rank 첨가행렬: ___   (미지수 3개)
   - 해의 존재·유일성 판정: ___
   - 두 rank 가 같을 때 / 다를 때 / 같지만 n 보다 작을 때: ___

3. 행렬식: ___
   역행렬:
   ___
   - det 이 0 에 가까울 때 위험한 이유: ___
   - 실제 판정 기준: ___

4. 피벗팅 없을 때 오차 / 부분 피벗팅 오차 (eps = 1e-4): ___ / ___
   - eps 를 더 줄이면: ___
   - 유효숫자가 날아가는 지점: ___

5. solve vs 역행렬 곱셈 (500x500)
   - 시간 : ___ ms / ___ ms
   - 잔차 : ___ / ___
   - 조건수를 키웠을 때의 정확도 차이: ___
   - 결론: ___
"""
print(summary)